# 01 — Environment & Feasibility (NaturalVoice AI, Phase 0)

**Purpose (PRD §7 Phase 0):** verify GPU type + VRAM, Python/PyTorch/CUDA compatibility, install the baseline model's dependencies, load the pretrained checkpoint, and record model version + settings.

**Cost discipline (PRD §11):**
- Run this notebook top-to-bottom ONCE per session, then shut the GPU runtime down.
- This notebook does NOT generate audio (that is `02_first_generation.ipynb`) — keep the feasibility check short.
- Do not proceed to the paid/long phases (fine-tuning, benchmarks) until this notebook passes and is committed.

**Model swap note:** the PRD's candidate baseline is 'VoxCPM2' (2B). At build time it could not be verified, so this notebook targets the published OpenBMB VoxCPM checkpoint. Swap points: `configs/baseline.yaml` (`model.id`, `revision`) and `app/tts_engine.py` ('MODEL API ADAPTER' functions).

In [ ]:
import json, platform, shutil, sys, os

env_info = {
    'python': platform.python_version(),
    'platform': platform.platform(),
    'cwd': os.getcwd(),
}
print(json.dumps(env_info, indent=2))

In [ ]:
import torch, yaml

gpu_info = {
    'torch': torch.__version__,
    'cuda_build': torch.version.cuda,
    'cuda_available': torch.cuda.is_available(),
}
if torch.cuda.is_available():
    prop = torch.cuda.get_device_properties(0)
    gpu_info.update({
        'gpu_name': prop.name,
        'vram_gib': round(prop.total_memory / 1024**3, 2),
        'cuda_capability': f'{prop.major}.{prop.minor}',
    })
    print(f"GPU: {prop.name} | VRAM: {gpu_info['vram_gib']} GiB | capability {gpu_info['cuda_capability']}")
else:
    gpu_info['vram_gib'] = 0
    print('WARNING: no CUDA GPU visible — switch the runtime to a GPU instance before loading the model.')
print(json.dumps(gpu_info, indent=2))

min_vram = (yaml.safe_load(open('configs/baseline.yaml')) or {}).get('limits', {}).get('min_vram_gib', 6)
if gpu_info['vram_gib'] < min_vram:
    print(f"STOP (cost rule 2, PRD §11): need >= {min_vram} GiB VRAM per configs/baseline.yaml — upgrade the runtime before loading the model.")
else:
    print(f"VRAM check passed (>= {min_vram} GiB).")

In [ ]:
import shutil
total, used, free = shutil.disk_usage('.')
print(f'disk free: {free / 1024**3:.1f} GiB (total {total / 1024**3:.1f} GiB)')
if free < 30 * 1024**3:
    print('WARNING: under ~30 GiB free — checkpoint downloads + pip packages may not fit.')
else:
    print('storage check passed.')

In [ ]:
# PRD §10 reproducibility: install the pinned versions from requirements.txt.
# (On Colab torch/torchaudio CUDA builds come preinstalled — do NOT force-reinstall them.)
%pip install -q voxcpm==2.0.3 soundfile pyyaml huggingface_hub

In [ ]:
import os
import importlib.metadata as md
import voxcpm

print('voxcpm version:', md.version('voxcpm'))
if os.environ.get('HF_TOKEN'):
    print('Note: HF_TOKEN is set in this runtime — never commit or print it (PRD §10).')
else:
    print('No credentials loaded (public checkpoint). Nothing to leak (PRD §10).')

In [ ]:
# Exit criterion PRD §7: model + execution settings recorded.
from pathlib import Path
import datetime, torch
import importlib.metadata as md

record = {
    'timestamp': datetime.datetime.now(datetime.timezone.utc).strftime('%Y-%m-%dT%H:%M:%SZ'),
    'phase': 'environment_check',
    'python': platform.python_version(),
    'platform': platform.platform(),
    'torch': torch.__version__,
    'cuda_build': torch.version.cuda,
    'cuda_available': torch.cuda.is_available(),
    'gpu_name': gpu_info.get('gpu_name'),
    'vram_gib': gpu_info.get('vram_gib'),
    'voxcpm_version': md.version('voxcpm'),
    'disk_free_gib': round(shutil.disk_usage('.').free / 1024**3, 2),
}
gen = Path('.gen')
gen.mkdir(exist_ok=True)
out_path = gen / f"env_check-{datetime.datetime.now():%Y%m%d-%H%M%S}.json"
out_path.write_text(json.dumps(record, indent=2, default=str))
print('Recorded environment snapshot to', out_path)
print(json.dumps(record, indent=2, default=str))

## Phase 0 checkpoint (notebook 01 of 2)

| Phase 0 exit criterion (PRD §7) | Status here |
|---|---|
| Model loads successfully | ✅ verified in `02_first_generation.ipynb` (loads checkpoint) |
| Test audio is generated | ✅ `02_first_generation.ipynb` |
| Audio can be played / downloaded | ✅ `02_first_generation.ipynb` |
| Model version + settings recorded | ✅ `.gen/env_check-*.json` (this notebook) + `gen/run_log.jsonl` (notebook 02) |

**Before moving on (cost rules 4-5, PRD §11):** commit the `.gen/` snapshot, then shut down / disconnect the GPU runtime if you are not continuing straight into notebook 02.